In [3]:
from data.corpus_canciones import get_collection
from src.rag_utils import chunking_por_estrofa, generar_o_cargar_embeddings, crear_indice_faiss, cargar_modelo
from sentence_transformers import SentenceTransformer
import pandas as pd

collection = get_collection()
cursor = collection.find({}, {"_id": 0, "artista": 1, "nombre": 1, "letra": 1, "genero": 1, "anio": 1, "titulo": 1})
df_canciones = pd.DataFrame(list(cursor))
datos = df_canciones.to_dict('records')
fragmentos_procesados_A = chunking_por_estrofa(datos)
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
emb_A, datos_completos = generar_o_cargar_embeddings(
    fragmentos_procesados_A,
    "emb_por_estrofa",
    modelo_emb
)

indice_A = crear_indice_faiss(emb_A)
print("RAG listo.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1084.19it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generando 24285 embeddings por primera vez...


Batches: 100%|██████████| 759/759 [14:16<00:00,  1.13s/it]


Proceso terminado y guardado en: emb_por_estrofa.pkl
Índice FAISS creado: 24285 vectores, dimensión 384
RAG listo.


In [4]:
from src.chatbot_engine import HistoryBotEngine
bot = HistoryBotEngine(indice_faiss=indice_A, chunks=fragmentos_procesados_A, modelo_emb=modelo_emb)

Cargando clasificador fine-tuneado...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3215.57it/s]


Cargando generador...
Cargando google/flan-t5-base...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1757.75it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo Base listo.
HistoryBot listo.


In [5]:
respuesta, decada, chunks = bot.responder("What songs talk about heartbreak in the 90s?")
print(respuesta)

Resultados para: 'What songs talk about heartbreak in the 90s?'

Resultado #1 (Similitud: 0.2656)
Song: Dome | Artist: day dakri,Reerock,St. Hoov | Genre: hip hop | Year: 2023
Lyrics: ow I'm sittin' here high and alone yeah Head full of heartbreak songs...
--------------------------------------------------
Resultado #2 (Similitud: 0.2560)
Song: Bottom of a Heartbreak | Artist: NEEDTOBREATHE | Genre: pop | Year: 2020
Lyrics: e to hide Where you feel safe At the bottom of a heartbreak At the bottom of a heartbreak Sometimes I think that if I climb that hill I wouldn...
--------------------------------------------------
Resultado #3 (Similitud: 0.2444)
Song: Pity the Plight | Artist: Plan B,John Cooper Clarke | Genre: hip hop | Year: 2012
Lyrics:  it Chris Don't tell me to fucking allow it you don't fucking know me These are the tears of a thug like murky water Crying tears as clear as ...
--------------------------------------------------
Resultado #4 (Similitud: 0.2198)
Song: Cryin' | A

In [4]:
from src.chatbot_engine import HistoryBotEngine

bot = HistoryBotEngine(
    indice_faiss=indice_A,       # de tu compañero
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb
)

respuesta, decada, chunks = bot.responder("What songs talk about heartbreak in the 90s?")
print(respuesta)

Cargando clasificador de décadas...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1815.07it/s]


Cargando generador Flan-T5...
Generador: Flan-T5 local
HistoryBot listo.
Resultados para: 'What songs talk about heartbreak in the 90s?'

Resultado #1 (Similitud: 0.2656)
Song: Dome | Artist: day dakri,Reerock,St. Hoov | Genre: hip hop | Year: 2023
Lyrics: ow I'm sittin' here high and alone yeah Head full of heartbreak songs...
--------------------------------------------------
Resultado #2 (Similitud: 0.2560)
Song: Bottom of a Heartbreak | Artist: NEEDTOBREATHE | Genre: pop | Year: 2020
Lyrics: e to hide Where you feel safe At the bottom of a heartbreak At the bottom of a heartbreak Sometimes I think that if I climb that hill I wouldn...
--------------------------------------------------
Resultado #3 (Similitud: 0.2444)
Song: Pity the Plight | Artist: Plan B,John Cooper Clarke | Genre: hip hop | Year: 2012
Lyrics:  it Chris Don't tell me to fucking allow it you don't fucking know me These are the tears of a thug like murky water Crying tears as clear as ...
---------------------------

In [5]:
# Ver qué hay en los chunks que trae el RAG
from src.rag_utils import buscar_chunks_relevantes
resultados = buscar_chunks_relevantes(
    pregunta="songs about loneliness",
    indice_FAISS=indice_A,
    chunks=fragmentos_procesados_A,
    modelo=modelo_emb,
    top_k=5
)
for r in resultados:
    print(r["chunk"]["texto_ia"][:100])

Resultados para: 'songs about loneliness'

Resultado #1 (Similitud: 0.4024)
Song: Lost Friends | Artist: Middle Kids | Genre: pop | Year: 2018
Lyrics: Lonely is the sound when the truth hits the ground I lost all my friends that day I lost all my friends We were sitting 'round and I remember ...
--------------------------------------------------
Resultado #2 (Similitud: 0.4022)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: he vacant space The cried out tears and a never ending maze Oh I have found what only loneliness provides A strength within knowing I will fin...
--------------------------------------------------
Resultado #3 (Similitud: 0.3815)
Song: Map of the Problematique | Artist: Muse | Genre: rock | Year: 2006
Lyrics: Fear and panic in the air I want to be free from desolation and despair And I feel like everything I sow Is being swept away well I refuse to ...
--------------------------------------------------
Resultado #4 (Similitud: